# 09 — Model Monitoring & Drift Detection

This notebook sets up **Snowflake ML Model Monitors** on the SNOWHEALTH demand forecast model to track:
- **Prediction drift** — Are model outputs shifting over time? (PSI)
- **Feature drift** — Are input distributions changing? (KL Divergence)
- **Performance metrics** — RMSE, MAE against actual demand (when labels available)
- **Statistical metrics** — Counts, nulls, distribution summaries

We then **simulate drift** by injecting shifted data and query the monitor to see drift detected in real time.

```
                  ┌─────────────────┐
Prediction Log →  │  Model Monitor  │ → Drift / Performance Metrics
(source table)    │  (refreshes on  │ → Alerts (via SQL functions)
                  │   schedule)     │ → Snowsight Dashboard
                  └─────────────────┘
```

**Prerequisites:** Run notebooks 05 (model deployment) and 08 (inference service) first.

In [ ]:
# Session setup
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
import pandas as pd
import numpy as np
import time

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()

MODEL_NAME = cfg.SNOWHEALTH_MODEL_NAME
VERSION_NAME = "TUNED_V1"
MONITOR_NAME = "SNOWHEALTH_DEMAND_MONITOR"

print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}")
print(f"Model: {MODEL_NAME} v{VERSION_NAME}")

In [ ]:
# Step 1: Create a prediction source table with predictions + actuals
# The model monitor reads from this table to compute metrics
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
mv = reg.get_model(MODEL_NAME).version(VERSION_NAME)

# Get feature data and run batch predictions
feature_input_cols = [
    'UNIT_PRICE', 'PROMOTION_DISCOUNT_PCT', 'TEMPERATURE_F', 'FLU_INDEX',
    'COMPETITOR_PROXIMITY_MILES', 'DAYS_SUPPLY_ON_HAND',
    'MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS',
    'ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS',
    'LAG_7D_UNITS', 'LAG_14D_UNITS', 'LAG_30D_UNITS',
    'PRICE_TIMES_PROMO', 'INVENTORY_TURNOVER_RATIO', 'SEASONAL_DEMAND_INDEX',
]

features_table = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES")

# Run predictions through the model
pred_result = mv.run(
    features_table.select(feature_input_cols),
    function_name="predict"
)

# Build source table: features + prediction + actual + timestamp + segments
features_pdf = features_table.to_pandas()
pred_pdf = pred_result.to_pandas()

source_pdf = features_pdf[feature_input_cols + ['UNITS_SOLD', 'REGION', 'PRODUCT_CATEGORY', 'SKU_ID', 'STORE_ID']].copy()
source_pdf['PREDICTION'] = pred_pdf.iloc[:, -1].values.astype(float)
source_pdf['ACTUAL'] = source_pdf['UNITS_SOLD'].astype(float)
source_pdf['PREDICTION_TIMESTAMP'] = pd.Timestamp.now()
source_pdf['REGION'] = source_pdf['REGION'].astype(str)
source_pdf['PRODUCT_CATEGORY'] = source_pdf['PRODUCT_CATEGORY'].astype(str)

print(f"Source table: {len(source_pdf)} rows")
print(f"Prediction range: {source_pdf['PREDICTION'].min():.1f} - {source_pdf['PREDICTION'].max():.1f}")
print(f"Actual range: {source_pdf['ACTUAL'].min():.0f} - {source_pdf['ACTUAL'].max():.0f}")
print(f"Segments: {source_pdf['REGION'].nunique()} regions, {source_pdf['PRODUCT_CATEGORY'].nunique()} categories")

In [ ]:
# Save source table to Snowflake (the monitor reads from this)
SOURCE_TABLE = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_PREDICTION_LOG"
BASELINE_TABLE = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_PREDICTION_BASELINE"

snow_source = session.create_dataframe(source_pdf)
snow_source.write.mode("overwrite").save_as_table(SOURCE_TABLE)

# Also save baseline (same data initially — drift = 0)
snow_source.write.mode("overwrite").save_as_table(BASELINE_TABLE)

count = session.table(SOURCE_TABLE).count()
print(f"Source table: {SOURCE_TABLE} ({count} rows)")
print(f"Baseline table: {BASELINE_TABLE} ({count} rows)")
print(f"\nColumns:")
for row in session.sql(f"DESCRIBE TABLE {SOURCE_TABLE}").collect():
    rd = row.as_dict()
    name = list(rd.values())[0]
    dtype = list(rd.values())[1]
    print(f"  {name}: {dtype}")

In [ ]:
%%sql -r col_check
-- Verify timestamp column type (must be TIMESTAMP_NTZ for model monitor)
SELECT 
    COLUMN_NAME, DATA_TYPE 
FROM SNOWHEALTH_ML_DB.INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_SCHEMA = 'SNOWHEALTH_MODELS' 
    AND TABLE_NAME = 'SNOWHEALTH_PREDICTION_LOG' 
    AND COLUMN_NAME IN ('PREDICTION_TIMESTAMP', 'PREDICTION', 'ACTUAL', 'REGION')
ORDER BY COLUMN_NAME

In [ ]:
# Step 2: Create Model Monitor with drift detection and segmentation
# Drop existing monitor if present (safe for re-runs)
try:
    session.sql(f"DROP MODEL MONITOR IF EXISTS {MONITOR_NAME}").collect()
    print(f"Dropped existing monitor: {MONITOR_NAME}")
except:
    pass

monitor_sql = f"""
CREATE MODEL MONITOR {MONITOR_NAME} WITH
    MODEL = {MODEL_NAME}
    VERSION = '{VERSION_NAME}'
    FUNCTION = 'PREDICT'
    SOURCE = {SOURCE_TABLE}
    WAREHOUSE = COMPUTE_WH
    REFRESH_INTERVAL = '1 HOUR'
    AGGREGATION_WINDOW = '1 DAY'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTION')
    ACTUAL_SCORE_COLUMNS = ('ACTUAL')
    BASELINE = {BASELINE_TABLE}
    SEGMENT_COLUMNS = ('REGION', 'PRODUCT_CATEGORY')
"""

print("Creating model monitor:")
print(monitor_sql)
result = session.sql(monitor_sql).collect()
print(f"\nResult: {result}")

In [ ]:
# Step 3: Verify monitor is active
monitor_info = session.sql(f"DESC MODEL MONITOR {MONITOR_NAME}").collect()
print("Monitor Details:")
for row in monitor_info:
    rd = row.as_dict()
    for k, v in rd.items():
        print(f"  {k}: {v}")

# Also check via SHOW
monitors = session.sql(f"SHOW MODEL MONITORS IN SCHEMA {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print(f"\nActive monitors: {len(monitors)}")
for m in monitors:
    rd = m.as_dict()
    name_key = [k for k in rd if 'name' in k.lower()]
    state_key = [k for k in rd if 'state' in k.lower() or 'status' in k.lower()]
    if name_key:
        print(f"  {rd[name_key[0]]}: {rd[state_key[0]] if state_key else 'unknown'}")

---
## Simulate Drift

We inject shifted prediction data to trigger drift detection:
1. **Temperature shift** (+20F) — simulates seasonal change
2. **Demand spike** — predictions systematically higher
3. **Promo saturation** — all items on promotion

This creates a measurable gap between the baseline distribution and the drifted source data.

In [ ]:
# Step 4: Simulate drift by inserting shifted data into the source table
np.random.seed(42)

drifted_pdf = source_pdf.copy()

# Apply drift transformations
drifted_pdf['TEMPERATURE_F'] = drifted_pdf['TEMPERATURE_F'] + 20  # Climate shift
drifted_pdf['ROLLING_7D_AVG_UNITS'] = drifted_pdf['ROLLING_7D_AVG_UNITS'] * 1.4  # Demand spike
drifted_pdf['PROMOTION_DISCOUNT_PCT'] = 0.25  # All items on promo
drifted_pdf['PRICE_TIMES_PROMO'] = drifted_pdf['UNIT_PRICE'] * 0.25
drifted_pdf['SEASONAL_DEMAND_INDEX'] = drifted_pdf['SEASONAL_DEMAND_INDEX'] * 1.3

# Re-predict with drifted features to get shifted predictions
drifted_snow = session.create_dataframe(drifted_pdf[feature_input_cols])
drifted_preds = mv.run(drifted_snow, function_name="predict").to_pandas()

drifted_pdf['PREDICTION'] = drifted_preds.iloc[:, -1].values.astype(float)
# Actuals stay the same (this creates prediction vs actual mismatch = performance degradation)
drifted_pdf['PREDICTION_TIMESTAMP'] = pd.Timestamp.now() + pd.Timedelta(hours=1)

print("Drift Injection Summary:")
print(f"  Temperature: +20F shift")
print(f"  Rolling 7D avg: 1.4x multiplier")
print(f"  Promotion: all items set to 25%")
print(f"  Seasonal index: 1.3x multiplier")
print(f"\nPrediction shift:")
print(f"  Baseline mean: {source_pdf['PREDICTION'].mean():.1f}")
print(f"  Drifted mean:  {drifted_pdf['PREDICTION'].mean():.1f}")
print(f"  Shift: {((drifted_pdf['PREDICTION'].mean() / source_pdf['PREDICTION'].mean()) - 1) * 100:.1f}%")
print(f"\nPerformance impact:")
baseline_mae = abs(source_pdf['ACTUAL'] - source_pdf['PREDICTION']).mean()
drifted_mae = abs(drifted_pdf['ACTUAL'] - drifted_pdf['PREDICTION']).mean()
print(f"  Baseline MAE: {baseline_mae:.1f} units")
print(f"  Drifted MAE:  {drifted_mae:.1f} units")
print(f"  Degradation:  {((drifted_mae / baseline_mae) - 1) * 100:.0f}%")

In [ ]:
# Append drifted data to the source table (monitor will pick this up on next refresh)
drifted_snow_full = session.create_dataframe(drifted_pdf)
drifted_snow_full.write.mode("append").save_as_table(SOURCE_TABLE)

new_count = session.table(SOURCE_TABLE).count()
print(f"Source table now has {new_count} rows (baseline + drifted)")
print(f"Monitor will detect drift on next refresh (configured: every 1 hour)")
print(f"\nTo force immediate refresh, the monitor refreshes automatically.")
print(f"Check results in Snowsight: AI & ML → Models → {MODEL_NAME} → Monitors")

In [ ]:
# Step 5: Query drift metrics from the monitor
print("=" * 60)
print("QUERYING MODEL MONITOR METRICS")
print("=" * 60)

# Drift metrics (PSI = POPULATION_STABILITY_INDEX)
try:
    drift_sql = f"""
    SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        '{MONITOR_NAME}',
        'POPULATION_STABILITY_INDEX',
        'PREDICTION',
        '1 DAY',
        DATEADD(day, -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    """
    drift_results = session.sql(drift_sql).to_pandas()
    if len(drift_results) > 0:
        print("\nPrediction Drift (PSI):")
        print(drift_results.to_string(index=False))
    else:
        print("\nNo drift metrics yet — monitor is still processing its first refresh.")
except Exception as e:
    print(f"Drift query: {e}")

# Performance metrics
try:
    perf_sql = f"""
    SELECT * FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
        '{MONITOR_NAME}',
        'MAE',
        '1 DAY',
        DATEADD(day, -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    """
    perf_results = session.sql(perf_sql).to_pandas()
    if len(perf_results) > 0:
        print("\nPerformance (MAE):")
        print(perf_results.to_string(index=False))
    else:
        print("\nNo performance metrics yet — waiting for refresh.")
except Exception as e:
    print(f"Performance query: {e}")

print("\nNote: Metrics populate after the monitor completes its first refresh (~minutes).")
print("View in Snowsight: AI & ML → Models → SNOWHEALTH_DEMAND_FORECAST → Monitors")

In [ ]:
# Step 6: Query drift by segment (region-level)
print("=" * 60)
print("SEGMENT-LEVEL DRIFT ANALYSIS")
print("=" * 60)

regions = ['Northeast', 'Southeast', 'Midwest', 'South', 'West']
for region in regions:
    try:
        seg_sql = f"""
        SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            '{MONITOR_NAME}',
            'POPULATION_STABILITY_INDEX',
            'PREDICTION',
            '1 DAY',
            DATEADD(day, -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
            '{{"SEGMENTS": [{{"column": "REGION", "value": "{region}"}}]}}'
        ))
        """
        seg_result = session.sql(seg_sql).to_pandas()
        if len(seg_result) > 0:
            psi_val = seg_result.iloc[0, -1] if len(seg_result.columns) > 0 else 'N/A'
            print(f"  {region:<15} PSI: {psi_val}")
        else:
            print(f"  {region:<15} No data yet (pending refresh)")
    except Exception as e:
        print(f"  {region:<15} Pending refresh")

print("\nPSI Interpretation:")
print("  < 0.1  : No significant drift")
print("  0.1-0.2: Moderate drift — investigate")
print("  > 0.2  : Significant drift — retrain recommended")

In [ ]:
# Step 7: Manual drift analysis (immediate — doesn't wait for monitor refresh)
from scipy import stats

print("=" * 60)
print("IMMEDIATE DRIFT DETECTION (scipy KS-test)")
print("=" * 60)

baseline_preds = source_pdf['PREDICTION'].values
drifted_preds_arr = drifted_pdf['PREDICTION'].values

# Overall prediction drift
ks_stat, p_value = stats.ks_2samp(baseline_preds, drifted_preds_arr)
print(f"\nPrediction Distribution:")
print(f"  KS statistic: {ks_stat:.4f}")
print(f"  p-value:      {p_value:.2e}")
print(f"  Status:       {'DRIFT DETECTED' if p_value < 0.05 else 'No significant drift'}")

# Feature-level drift
print(f"\nFeature Drift (KS Test):")
print(f"  {'Feature':<30} {'KS Stat':>8} {'p-value':>12}  {'Status'}")
print(f"  {'-'*70}")
drift_features = ['TEMPERATURE_F', 'ROLLING_7D_AVG_UNITS', 'PROMOTION_DISCOUNT_PCT',
                  'PRICE_TIMES_PROMO', 'SEASONAL_DEMAND_INDEX', 'UNIT_PRICE',
                  'FLU_INDEX', 'DAYS_SUPPLY_ON_HAND']
for feat in drift_features:
    ks, pv = stats.ks_2samp(source_pdf[feat].values, drifted_pdf[feat].values)
    flag = 'DRIFT' if pv < 0.05 else 'OK'
    print(f"  {feat:<30} {ks:>8.4f} {pv:>12.2e}  {flag}")

In [ ]:
# Visualization: Baseline vs Drifted distributions
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_features = ['PREDICTION', 'TEMPERATURE_F', 'ROLLING_7D_AVG_UNITS',
                 'PROMOTION_DISCOUNT_PCT', 'PRICE_TIMES_PROMO', 'SEASONAL_DEMAND_INDEX']

for i, feat in enumerate(plot_features):
    ax = axes[i // 3][i % 3]
    ax.hist(source_pdf[feat], bins=40, alpha=0.5, color='#2196F3', label='Baseline', density=True)
    ax.hist(drifted_pdf[feat], bins=40, alpha=0.5, color='#FF5722', label='Drifted', density=True)
    ks, pv = stats.ks_2samp(source_pdf[feat].values, drifted_pdf[feat].values)
    ax.set_title(f'{feat}\nKS={ks:.3f}, p={pv:.1e}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

plt.suptitle('Baseline vs Drifted Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Monitor management reference
print("=" * 60)
print("MODEL MONITOR MANAGEMENT")
print("=" * 60)
print(f"""
View in Snowsight:
  AI & ML → Models → {MODEL_NAME} → Monitors

Check status:
  DESC MODEL MONITOR {MONITOR_NAME};
  SHOW MODEL MONITORS IN SCHEMA {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA};

Query drift (PSI):
  SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
      '{MONITOR_NAME}', 'PSI', 'PREDICTION', 'DAY',
      DATEADD(day, -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
      CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
  ));

Query performance (MAE/RMSE):
  SELECT * FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
      '{MONITOR_NAME}', 'MAE', 'DAY',
      DATEADD(day, -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
      CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
  ));

Suspend/Resume:
  ALTER MODEL MONITOR {MONITOR_NAME} SUSPEND;
  ALTER MODEL MONITOR {MONITOR_NAME} RESUME;

Drop monitor:
  DROP MODEL MONITOR {MONITOR_NAME};
""")

print("Notebook complete. The monitor is active and will detect drift on its refresh schedule.")
print("Drifted data has been injected — check Snowsight for results after the first refresh.")